# Capítulo 5 — Visualización con Matplotlib y Seaborn

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### Introducción del capítulo

In [ ]:
!pip install -q pandas==2.3.3

### 5.1 Anatomía de Matplotlib: `Figure`, `Axes` y la API orientada a objetos

#### Las dos APIs, y por qué solo una es profesional

In [ ]:
import matplotlib.pyplot as plt

x = [1, 2, 3, 4]
y = [120, 135, 128, 150]

# API de estado: NO recomendada para trabajo serio
plt.plot(x, y)
plt.title("Ventas")      # ¿a qué gráfico se aplica? Al "actual", sea cual sea
plt.show()

In [ ]:
# devuelve la Figure y el Axes, ambos como variables
fig, ax = plt.subplots()
ax.plot(x, y)                 # se dibuja SOBRE ese Axes concreto
ax.set_title("Ventas")        # se titula ESE Axes, sin ambigüedad
fig.tight_layout()            # se ajusta ESA figura

*Esquema del libro (no se ejecuta):*

```python
fig, ejes = plt.subplots(2, 2, figsize=(10, 7.5))   # rejilla de 2x2 -> 4 Axes
ejes[0, 0].plot(...)      # se accede a cada uno por su posición en la rejilla
ejes[1, 1].hist(...)
```

### 5.2 Gráficos fundamentales: qué pregunta responde cada uno

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Datos del capítulo, publicados en el repositorio del libro
DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
ventas = pd.read_csv(DATOS + "ventas_cap5.csv")
serie_mensual = pd.Series(
    [120, 135, 128, 150, 172, 165, 190, 205, 198, 220, 240, 265],
    index=pd.date_range("2024-01-31", periods=12, freq="ME"),
)

fig, ejes = plt.subplots(2, 2, figsize=(10, 7.5))

# LÍNEA: evolución de una magnitud a lo largo del tiempo
ejes[0, 0].plot(serie_mensual.index, serie_mensual.values, marker="o",
                color="#1F3864")
ejes[0, 0].set_title("Línea: ¿cómo evoluciona en el tiempo?",
                     fontweight="bold", fontsize=10)
ejes[0, 0].set_ylabel("Ingreso (miles)")
ejes[0, 0].tick_params(axis="x", rotation=45)
ejes[0, 0].grid(alpha=0.3)

# BARRAS: comparación de una magnitud entre categorías
por_categoria = (ventas.groupby("categoria")["ingreso"].sum()
                 .sort_values() / 1000)
ejes[0, 1].barh(por_categoria.index, por_categoria.values, color="#2E75B6")
ejes[0, 1].set_title("Barras: ¿cómo se comparan las categorías?",
                     fontweight="bold", fontsize=10)
ejes[0, 1].set_xlabel("Ingreso total (miles)")
ejes[0, 1].grid(alpha=0.3, axis="x")

# DISPERSIÓN: relación entre dos variables numéricas
ejes[1, 0].scatter(ventas["unidades"], ventas["ingreso"], alpha=0.5, s=22,
                   color="#1F3864")
ejes[1, 0].set_title("Dispersión: ¿hay relación entre dos variables?",
                     fontweight="bold", fontsize=10)
ejes[1, 0].set_xlabel("Unidades")
ejes[1, 0].set_ylabel("Ingreso")
ejes[1, 0].grid(alpha=0.3)

# HISTOGRAMA: distribución de una sola variable numérica
ejes[1, 1].hist(ventas["precio"], bins=30, color="#2E75B6", edgecolor="white")
ejes[1, 1].set_title("Histograma: ¿cómo se distribuye una variable?",
                     fontweight="bold", fontsize=10)
ejes[1, 1].set_xlabel("Precio")
ejes[1, 1].set_ylabel("Frecuencia")
ejes[1, 1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("fig_5_fundamentales.png", dpi=160, bbox_inches="tight")

### 5.3 Personalización: títulos, ejes, leyendas, anotaciones, escalas y subplots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

serie = pd.Series(
    [120, 135, 128, 150, 172, 165, 190, 205, 198, 220, 240, 265],
    index=pd.date_range("2024-01-31", periods=12, freq="ME"),
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(serie.index, serie.values, marker="o", color="#1F3864", label="2024")

ax.set_title("Ingreso mensual 2024", fontsize=13, fontweight="bold")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingreso (miles de euros)")     # SIEMPRE con unidades
ax.legend(loc="upper left")
# rejilla tenue: ayuda a leer, no compite
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=45)

In [ ]:
fecha_pico, valor_pico = serie.idxmax(), serie.max()
ax.annotate("Máximo anual",
            xy=(fecha_pico, valor_pico),              # punto señalado
            xytext=(fecha_pico, valor_pico * 1.15),   # posición del texto
            arrowprops=dict(arrowstyle="->", color="#C00000"),
            ha="center", color="#C00000", fontweight="bold");

In [ ]:
ax.set_yscale("log")

In [ ]:
fig.savefig("grafico.png", dpi=300, bbox_inches="tight")

### 5.4 Seaborn: integración con DataFrames y gráficos estadísticos de alto nivel

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
ventas = pd.read_csv(DATOS + "ventas_cap5.csv")

# Con Matplotlib puro: hay que separar los grupos a mano
fig, ax = plt.subplots()
for categoria in ventas["categoria"].unique():
    subconjunto = ventas[ventas["categoria"] == categoria]
    ax.hist(subconjunto["precio"], alpha=0.5, label=categoria)
ax.legend()

# Con Seaborn: la agrupación se declara
fig, ax = plt.subplots()
sns.histplot(data=ventas, x="precio", hue="categoria", kde=True, ax=ax);

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
sns.boxplot(data=ventas, x="categoria", y="precio", ax=ejes[1])
ejes[1].set_title("Precio por categoría");    # personalización de Matplotlib

In [ ]:
sns.set_theme(style="whitegrid")

### 5.5 Distribuciones y relaciones: `histplot`, `boxplot`, `violinplot`, `scatterplot`, `pairplot`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
ventas = pd.read_csv(DATOS + "ventas_cap5.csv")
sns.set_theme(style="whitegrid")

fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
sns.histplot(data=ventas, x="precio", hue="categoria", kde=True, ax=ejes[0],
             palette="deep")
ejes[0].set_title("histplot: distribución por grupo", fontweight="bold",
                  fontsize=10)
sns.boxplot(data=ventas, x="categoria", y="precio", ax=ejes[1],
            palette="deep", hue="categoria", legend=False)
ejes[1].set_title("boxplot: resumen de cinco números", fontweight="bold",
                  fontsize=10)
sns.violinplot(data=ventas, x="categoria", y="precio", ax=ejes[2],
               palette="deep", hue="categoria", legend=False)
ejes[2].set_title("violinplot: forma completa", fontweight="bold",
                  fontsize=10)
for eje in ejes[1:]:
    eje.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig("fig_5_seaborn.png", dpi=160, bbox_inches="tight")

In [ ]:
sns.pairplot(ventas, hue="categoria", vars=["precio", "unidades", "ingreso"]);

### 5.6 Mapas de calor, correlación, y por qué visualizar no es opcional

#### Pearson frente a Spearman

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
puntos = pd.read_csv(DATOS + "correlacion_cap5.csv")
sns.set_theme(style="whitegrid")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
paneles = [("x_curva", "y_curva", "#1F3864", "Relación monótona no lineal"),
           ("x_recta", "y_recta", "#C00000", "Relación lineal")]
for eje, (col_x, col_y, color, titulo) in zip(ejes, paneles):
    r_p, _ = stats.pearsonr(puntos[col_x], puntos[col_y])
    r_s, _ = stats.spearmanr(puntos[col_x], puntos[col_y])
    eje.scatter(puntos[col_x], puntos[col_y], s=28, color=color, alpha=0.75)
    eje.set_title(f"{titulo}\nPearson = {r_p:.2f}   |   Spearman = {r_s:.2f}",
                  fontweight="bold", fontsize=10)
    eje.set_xlabel("x"); eje.set_ylabel("y"); eje.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_5_pearson_spearman.png", dpi=160, bbox_inches="tight")

#### El mapa de calor

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
ventas = pd.read_csv(DATOS + "ventas_cap5.csv")
sns.set_theme(style="whitegrid")

numericas = ventas[["unidades", "precio", "ingreso", "descuento"]]
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
for eje, metodo, titulo in zip(ejes, ["pearson", "spearman"],
                               ["Correlación de Pearson (lineal)",
                                "Correlación de Spearman (por rangos)"]):
    matriz = numericas.corr(method=metodo)
    sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=eje, square=True, linewidths=0.5)
    eje.set_title(titulo, fontweight="bold", fontsize=10)
fig.tight_layout()
fig.savefig("fig_5_heatmap.png", dpi=160, bbox_inches="tight")

#### El cuarteto de Anscombe

In [ ]:
import pandas as pd
from scipy import stats

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
anscombe = pd.read_csv(DATOS + "anscombe.csv")

filas = []
for nombre, grupo in anscombe.groupby("conjunto"):
    pendiente, intercepto, r, _, _ = stats.linregress(grupo["x"], grupo["y"])
    filas.append({
        "conjunto": nombre,
        "media_x": round(grupo["x"].mean(), 2),
        "media_y": round(grupo["y"].mean(), 2),
        "var_x": round(grupo["x"].var(), 2),
        "var_y": round(grupo["y"].var(), 2),
        "pearson": round(r, 3),
        "recta": f"y = {intercepto:.2f} + {pendiente:.2f}x",
    })
print(pd.DataFrame(filas).to_string(index=False))

*Salida en el libro:*

```text
conjunto  media_x  media_y  var_x  var_y  pearson            recta
       I      9.0      7.5   11.0   4.13    0.816 y = 3.00 + 0.50x
      II      9.0      7.5   11.0   4.13    0.816 y = 3.00 + 0.50x
     III      9.0      7.5   11.0   4.12    0.816 y = 3.00 + 0.50x
      IV      9.0      7.5   11.0   4.12    0.817 y = 3.00 + 0.50x
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, ejes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
for eje, (nombre, grupo) in zip(ejes.flat, anscombe.groupby("conjunto")):
    eje.scatter(grupo["x"], grupo["y"], s=45, color="#1F3864", zorder=3,
                edgecolor="white")
    pendiente, intercepto, r, _, _ = stats.linregress(grupo["x"], grupo["y"])
    dominio = np.array([2, 20])
    eje.plot(dominio, intercepto + pendiente * dominio, color="#C00000",
             linewidth=1.6, zorder=2)
    eje.set_title(f"Conjunto {nombre}   (r = {r:.2f})", fontsize=11,
                  fontweight="bold")
    eje.grid(alpha=0.3, zorder=1)
    eje.set_xlim(2, 20)
    eje.set_ylim(2, 14)
    eje.set_xticks(range(2, 21, 2))
fig.suptitle("El cuarteto de Anscombe: estadísticos idénticos, "
             "realidades distintas", fontsize=13, fontweight="bold")
fig.supxlabel("x")
fig.supylabel("y")
fig.tight_layout()
fig.savefig("fig_5_anscombe.png", dpi=160, bbox_inches="tight")

### 5.7 Ética visual: la responsabilidad de no mentir con gráficos

#### El eje truncado

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
empresas = ["A", "B", "C", "D"]
satisfaccion = [92.0, 93.5, 94.0, 95.2]

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
ejes[0].bar(empresas, satisfaccion, color="#C00000")
ejes[0].set_ylim(91, 96)
ejes[0].set_title("ENGAÑOSO: eje truncado (empieza en 91)",
                  fontweight="bold", fontsize=10, color="#C00000")
ejes[0].set_ylabel("Satisfacción (%)")
ejes[1].bar(empresas, satisfaccion, color="#2E75B6")
ejes[1].set_ylim(0, 100)
ejes[1].set_title("HONESTO: eje desde cero", fontweight="bold", fontsize=10,
                  color="#1F3864")
ejes[1].set_ylabel("Satisfacción (%)")
for eje in ejes:
    eje.grid(alpha=0.3, axis="y")
fig.suptitle("Los mismos datos, dos mensajes opuestos", fontsize=12,
             fontweight="bold")
fig.tight_layout()
fig.savefig("fig_5_etica.png", dpi=160, bbox_inches="tight")

## Soluciones de los ejercicios

### [Ex-L2-C5-01] Panel de cuatro preguntas de negocio

#### Desarrollo con código

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Ventas de dos años con semilla fija: cuatro categorías de precio muy
# distinto
generador = np.random.default_rng(2024)
N = 1200
ventas = pd.DataFrame({
    "fecha": pd.to_datetime("2023-01-01")
             + pd.to_timedelta(generador.integers(0, 730, N), unit="D"),
    "categoria": generador.choice(["Electrónica", "Hogar", "Ropa",
                                   "Deportes"], N,
                                  p=[0.30, 0.28, 0.24, 0.18]),
    "region": generador.choice(["Norte", "Sur", "Este"], N),
    "unidades": generador.integers(1, 25, N),
})
ventas["precio"] = np.select(
    [ventas["categoria"] == "Electrónica", ventas["categoria"] == "Hogar",
     ventas["categoria"] == "Ropa"],
    [generador.normal(240, 65, N), generador.normal(95, 28, N),
     generador.normal(45, 15, N)],
    default=generador.normal(130, 40, N)).clip(5).round(2)
ventas["ingreso"] = (ventas["precio"] * ventas["unidades"]).round(2)

# Las cuatro preguntas se formulan ANTES de escribir una sola línea de
# gráfico:
#   P1. ¿Cómo evoluciona el ingreso mensual?
#       -> LÍNEA (eje X temporal)
#   P2. ¿Qué categorías aportan más ingreso?
#       -> BARRAS (comparar categorías)
#   P3. ¿El volumen explica el ingreso?
#       -> DISPERSIÓN (dos numéricas)
#   P4. ¿Cómo se distribuye el ticket por región?
#       -> BOXPLOT (distribución por grupo)

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))

# --- P1: LÍNEA. El eje X tiene orden natural (tiempo): la línea es el gráfico
# correcto.
serie = ventas.set_index("fecha")["ingreso"].resample("ME").sum() / 1000
ejes[0, 0].plot(serie.index, serie.values, marker="o", color="#1F3864",
                linewidth=1.6)
ejes[0, 0].set_title("P1. ¿Cómo evoluciona el ingreso mensual?",
                     fontweight="bold", fontsize=10)
ejes[0, 0].set_xlabel("Mes")
ejes[0, 0].set_ylabel("Ingreso (miles de €)")      # unidades SIEMPRE
ejes[0, 0].tick_params(axis="x", rotation=45)

# Anotación: dirigir la atención al punto relevante (Sección 5.3)
maximo = serie.idxmax()
ejes[0, 0].annotate(f"Máximo: {serie.max():.0f}k",
                    xy=(maximo, serie.max()),
                    xytext=(maximo, serie.max() * 1.12),
                    arrowprops=dict(arrowstyle="->", color="#C00000"),
                    ha="center", color="#C00000", fontsize=8,
                    fontweight="bold")

# --- P2: BARRAS horizontales. Etiquetas largas -> barh evita rotarlas.
por_categoria = (ventas.groupby("categoria")["ingreso"].sum().sort_values()
                 / 1000)
ejes[0, 1].barh(por_categoria.index, por_categoria.values, color="#2E75B6")
ejes[0, 1].set_title("P2. ¿Qué categorías aportan más ingreso?",
                     fontweight="bold", fontsize=10)
ejes[0, 1].set_xlabel("Ingreso total (miles de €)")
# valor sobre cada barra
for indice, valor in enumerate(por_categoria.values):
    ejes[0, 1].text(valor, indice, f" {valor:.0f}", va="center", fontsize=8)

# --- P3: DISPERSIÓN con alpha. Dos variables numéricas y muchos puntos
# superpuestos.
for categoria, color in zip(["Electrónica", "Hogar", "Ropa", "Deportes"],
                            ["#1F3864", "#2E75B6", "#C00000", "#548235"]):
    subconjunto = ventas[ventas["categoria"] == categoria]
    ejes[1, 0].scatter(subconjunto["unidades"], subconjunto["ingreso"],
                       alpha=0.45, s=18, label=categoria, color=color)
ejes[1, 0].set_title("P3. ¿El volumen explica el ingreso?", fontweight="bold",
                     fontsize=10)
ejes[1, 0].set_xlabel("Unidades por venta")
ejes[1, 0].set_ylabel("Ingreso (€)")
ejes[1, 0].legend(fontsize=7, loc="upper left")

# --- P4: BOXPLOT. La pregunta es por la DISTRIBUCIÓN, no por el promedio.
sns.boxplot(data=ventas, x="region", y="ingreso", ax=ejes[1, 1],
            hue="region", legend=False, palette="deep")
ejes[1, 1].set_title("P4. ¿Cómo se distribuye el ticket por región?",
                     fontweight="bold", fontsize=10)
ejes[1, 1].set_xlabel("Región")
ejes[1, 1].set_ylabel("Ingreso por venta (€)")

fig.tight_layout()
fig.savefig("fig_sol_c5_01.png", dpi=300, bbox_inches="tight")

#### Salida esperada

In [ ]:
correlacion = ventas["unidades"].corr(ventas["ingreso"])
print(f"P1 - ingreso mensual: de {serie.min():.0f}k a {serie.max():.0f}k")
print(f"P2 - categoría líder: {por_categoria.idxmax()} "
      f"({por_categoria.max():.0f}k, "
      f"{por_categoria.max() / por_categoria.sum() * 100:.1f}% del total)")
print(f"P3 - correlación unidades-ingreso: {correlacion:.3f}")
print("P4 - ticket mediano por región:")
print(ventas.groupby("region")["ingreso"].median().round(2))

*Salida en el libro:*

```text
P1 - ingreso mensual: de 59k a 114k
P2 - categoría líder: Electrónica (978k, 50.5% del total)
P3 - correlación unidades-ingreso: 0.591
P4 - ticket mediano por región:
region
Este     1075.16
Norte    1227.60
Sur      1181.34
Name: ingreso, dtype: float64
```

### [Ex-L2-C5-02] Auditoría de gráficos engañosos

#### Desarrollo con código

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, ejes = plt.subplots(2, 3, figsize=(13, 7.5))

# ---------------------------------------------
# MANIPULACIÓN 1 — EJE TRUNCADO
# ---------------------------------------------
empresas = ["Nosotros", "Rival A", "Rival B", "Rival C"]
cuota = [24.5, 23.8, 23.2, 22.9]        # cuotas casi idénticas

ejes[0, 0].bar(empresas, cuota, color="#C00000")
# LA MANIPULACIÓN: el eje Y empieza en 22.5
ejes[0, 0].set_ylim(22.5, 25)
ejes[0, 0].set_title("ENGAÑOSO 1: eje truncado en 22.5", fontweight="bold",
                     fontsize=9, color="#C00000")
ejes[1, 0].bar(empresas, cuota, color="#2E75B6")
# La corrección: eje desde cero
ejes[1, 0].set_ylim(0, 30)
ejes[1, 0].set_title("HONESTO 1: eje desde cero", fontweight="bold",
                     fontsize=9, color="#1F3864")
for eje in ejes[:, 0]:
    eje.set_ylabel("Cuota de mercado (%)")
    eje.tick_params(axis="x", rotation=15)

# ---------------------------------------------
# MANIPULACIÓN 2 — ESCALA LOGARÍTMICA NO DECLARADA
# ---------------------------------------------
paises = ["A", "B", "C", "D"]
casos = [50, 500, 5000, 50000]          # cada uno 10 veces el anterior

ejes[0, 1].bar(paises, casos, color="#C00000")
ejes[0, 1].set_yscale("log")            # <- LA MANIPULACIÓN (sin indicarlo)
# las barras arrancan en 1 = 10^0
ejes[0, 1].set_ylim(bottom=1)
ejes[0, 1].set_title("ENGAÑOSO 2: escala log sin declarar", fontweight="bold",
                     fontsize=9, color="#C00000")
# La corrección: escala lineal
ejes[1, 1].bar(paises, casos, color="#2E75B6")
ejes[1, 1].set_title("HONESTO 2: escala lineal", fontweight="bold",
                     fontsize=9, color="#1F3864")
for eje in ejes[:, 1]:
    eje.set_ylabel("Casos")

# ---------------------------------------------
# MANIPULACIÓN 3 — RECORTE DEL RANGO TEMPORAL
# ---------------------------------------------
# Serie de 24 meses en descenso, con un repunte en los últimos 6.
meses = pd.date_range("2023-01-31", periods=24, freq="ME")
generador = np.random.default_rng(11)
tendencia = np.linspace(100, 62, 24) + generador.normal(0, 3, 24)
tendencia[18:] = tendencia[18] + np.linspace(0, 9, 6)
serie_completa = pd.Series(tendencia, index=meses)
recorte = serie_completa.iloc[18:]      # LA MANIPULACIÓN: solo el repunte

ejes[0, 2].plot(recorte.index, recorte.values, marker="o", color="#C00000",
                linewidth=2)
ejes[0, 2].set_title("ENGAÑOSO 3: solo los últimos 6 meses",
                     fontweight="bold", fontsize=9, color="#C00000")
ejes[1, 2].plot(serie_completa.index, serie_completa.values, marker="o",
                color="#2E75B6", markersize=3)
# La corrección sombrea el tramo que el gráfico engañoso mostraba en solitario
ejes[1, 2].axvspan(recorte.index[0], recorte.index[-1], alpha=0.18,
                   color="#C00000")
ejes[1, 2].set_title("HONESTO 3: serie completa (24 meses)",
                     fontweight="bold", fontsize=9, color="#1F3864")
for eje in ejes[:, 2]:
    eje.set_ylabel("Usuarios activos (miles)")
    eje.tick_params(axis="x", rotation=45)

fig.suptitle("Auditoría de gráficos engañosos: arriba la versión manipulada, "
             "abajo la honesta", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig("fig_sol_c5_02.png", dpi=160, bbox_inches="tight")

#### Salida esperada

In [ ]:
diferencia_real = (cuota[0] - cuota[-1]) / cuota[-1] * 100
altura_aparente = (cuota[0] - 22.5) / (cuota[-1] - 22.5)
print("MANIPULACIÓN 1 (eje truncado)")
print(f"  diferencia real Nosotros vs Rival C : {diferencia_real:.1f}%")
print(f"  altura aparente de la barra         : {altura_aparente:.1f}x")
print(f"  factor de exageración               : "
      f"{altura_aparente / (1 + diferencia_real / 100):.1f}x")

pie = np.log10(ejes[0, 1].get_ylim()[0])
razon_real = casos[-1] / casos[0]
razon_percibida = (np.log10(casos[-1]) - pie) / (np.log10(casos[0]) - pie)
print("\nMANIPULACIÓN 2 (escala logarítmica no declarada)")
print(f"  razón real D/A            : {razon_real:.0f}x")
print(f"  razón de altura percibida : {razon_percibida:.1f}x")
print(f"  subestimación             : {razon_real / razon_percibida:.0f} "
      f"veces")

cambio_mostrado = (recorte.iloc[-1] - recorte.iloc[0]) / recorte.iloc[0] * 100
cambio_real = ((serie_completa.iloc[-1] - serie_completa.iloc[0])
               / serie_completa.iloc[0] * 100)
print("\nMANIPULACIÓN 3 (recorte del rango temporal)")
print(f"  cambio en los 6 meses mostrados : {cambio_mostrado:+.1f}%")
print(f"  cambio en los 24 meses reales   : {cambio_real:+.1f}%")

*Salida en el libro:*

```text
MANIPULACIÓN 1 (eje truncado)
  diferencia real Nosotros vs Rival C : 7.0%
  altura aparente de la barra         : 5.0x
  factor de exageración               : 4.7x

MANIPULACIÓN 2 (escala logarítmica no declarada)
  razón real D/A            : 1000x
  razón de altura percibida : 2.8x
  subestimación             : 362 veces

MANIPULACIÓN 3 (recorte del rango temporal)
  cambio en los 6 meses mostrados : +12.9%
  cambio en los 24 meses reales   : -21.3%
```

### [Ex-L2-C5-03] Mapas de calor del proyecto de similitud

#### Desarrollo con código

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# Los mismos vocabularios del Ejercicio [Ex-L2-C2-03]
idiomas = {
    "Español":   {"casa", "sol", "luna", "agua", "noche", "mano", "libro",
                  "flor", "mar", "luz"},
    "Inglés":    {"house", "sun", "moon", "water", "night", "hand", "book",
                  "flower", "sea", "light"},
    "Portugués": {"casa", "sol", "lua", "agua", "noite", "mao", "livro",
                  "flor", "mar", "luz"},
    "Francés":   {"maison", "soleil", "lune", "eau", "nuit", "main", "livre",
                  "fleur", "mer", "lumiere"},
}
nombres = list(idiomas.keys())


def jaccard(a, b):
    """J(A, B) = |A ∩ B| / |A ∪ B|."""
    union = len(a | b)
    return len(a & b) / union if union else 1.0


# Vectorización de los idiomas (del Ejercicio [Ex-L2-C2-03])
vocabulario = sorted(set().union(*idiomas.values()))
matriz_binaria = np.array([[1 if p in idiomas[i] else 0 for p in vocabulario]
                           for i in nombres])

# Similitud coseno de todos los pares en una sola operación matricial
normas = np.linalg.norm(matriz_binaria, axis=1, keepdims=True)
matriz_coseno = (matriz_binaria / normas) @ (matriz_binaria / normas).T

# Índice de Jaccard, para comparar
matriz_jaccard = np.array([[jaccard(idiomas[a], idiomas[b]) for b in nombres]
                           for a in nombres])

fig, ejes = plt.subplots(1, 3, figsize=(15, 4.4))

# Paneles 1 y 2: MISMA escala (vmin=0, vmax=1) para que los colores sean
# comparables.
# Sin fijarla, cada mapa se reescalaría a su propio rango y el contraste
# engañaría.
for eje, matriz, titulo in zip(ejes[:2], [matriz_jaccard, matriz_coseno],
                               ["Índice de Jaccard (Libro 1)",
                                "Similitud coseno (Cap. 2)"]):
    sns.heatmap(matriz, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1,
                xticklabels=nombres, yticklabels=nombres, square=True,
                linewidths=0.5, ax=eje, cbar_kws={"shrink": 0.75})
    eje.set_title(titulo, fontweight="bold", fontsize=10)
    eje.tick_params(axis="x", rotation=30)

# Panel 3: la DIFERENCIA. Paleta divergente centrada en cero, porque el signo
# importa.
# Redondeo: quita el ruido de punto flotante (del orden de 1e-16, que depende
# del procesador) y el signo de -0.0, para que un cero se vea como 0.000
diferencia = np.round(matriz_coseno - matriz_jaccard, 10) + 0.0
sns.heatmap(diferencia, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            vmin=-0.2, vmax=0.2, xticklabels=nombres, yticklabels=nombres,
            square=True, linewidths=0.5, ax=ejes[2],
            cbar_kws={"shrink": 0.75})
ejes[2].set_title("Diferencia (coseno − Jaccard)", fontweight="bold",
                  fontsize=10)
ejes[2].tick_params(axis="x", rotation=30)

fig.suptitle("Dos métricas sobre los mismos vocabularios: misma estructura, "
             "distinta magnitud", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig("fig_sol_c5_03.png", dpi=160, bbox_inches="tight")

#### Salida esperada

In [ ]:
fuera_de_diagonal = ~np.eye(len(nombres), dtype=bool)
ceros = (matriz_jaccard == 0) & (matriz_coseno == 0) & fuera_de_diagonal
print(f"Jaccard  español-portugués : {matriz_jaccard[0, 2]:.3f}")
print(f"Coseno   español-portugués : {matriz_coseno[0, 2]:.3f}")
print(f"Diferencia máxima fuera de la diagonal : "
      f"{np.abs(diferencia[fuera_de_diagonal]).max():.3f}")
print(f"Pares con valor 0.000 en AMBAS métricas: {ceros.sum() // 2} de 6")

*Salida en el libro:*

```text
Jaccard  español-portugués : 0.429
Coseno   español-portugués : 0.600
Diferencia máxima fuera de la diagonal : 0.171
Pares con valor 0.000 en AMBAS métricas: 5 de 6
```